# Experiment 03: Triage Gate Architecture Progression & Feature Pooling Ablation Suite

This study documents the evolution of the **Reflective-CXR Decision Engine**:
1. **Mathematical Formulation**: Decoupled cost-sensitive triage strictly bounding False Negative Rates (FNR) via calibrated probabilities.
2. **Linear Baseline**: CLS token linear probing using L2-regularized Logistic Regression.
3. **Representation Pooling Slicing**: Systematic ablation of spatial token pooling — CLS (0:768), Mean-Pool (768:1536), and Max-Pool (1536:2304).
4. **Clinical Operating Tradeoff**: Sensitivity curve analyzing dataset triage coverage rate versus False Negative Rate.


## Section 1: Decoupled Risk Calibration & Triage Objective


Reflective-CXR: Risk-Calibrated Clinical Triage
Objective: A decoupled framework to safely automate chest radiography triage while strictly bounding false negative rates using Vision-Language Pretraining and Isotonic Calibration.


In [4]:
!pip install sacremoses
!pip install -q peft

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import lightgbm as lgb
from tqdm import tqdm
from sklearn.calibration import IsotonicRegression
from sklearn.metrics import brier_score_loss

# CORRECTED IMPORT HERE:
from transformers import ViTModel, BioGptForCausalLM
from peft import LoraConfig, get_peft_model

# Check GPU availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using compute device: {device}")


Using compute device: cpu


Phase 1: Generative Vision-Language Pretraining (VLP)
We construct a multimodal architecture linking a Vision Transformer (ViT-Base) to a BioGPT decoder. To prevent catastrophic forgetting and ensure efficient training, the ViT encoder is adapted using Low-Rank Adaptation (LoRA).


In [5]:
class ReflectiveVLP(nn.Module):
    def __init__(self, vit_model_name='google/vit-base-patch16-224-in21k', lora_rank=8):
        super(ReflectiveVLP, self).__init__()
        
        # Initialize Base Vision Encoder
        print("Loading base ViT encoder...")
        self.encoder = ViTModel.from_pretrained(vit_model_name)
        
        # Apply LoRA to the Encoder (W' = W + BA)
        lora_config = LoraConfig(
            r=lora_rank,
            lora_alpha=16,
            target_modules=["query", "value"], 
            lora_dropout=0.1,
            bias="none"
        )
        self.encoder = get_peft_model(self.encoder, lora_config)
        
        # Initialize Text Decoder
        print("Loading BioGPT decoder...")
        
        # CORRECTED CLASS USAGE HERE:
        self.decoder = BioGptForCausalLM.from_pretrained("microsoft/biogpt")
        
        # Projection layer to align dimensions
        self.proj = nn.Linear(self.encoder.config.hidden_size, self.decoder.config.hidden_size)

    def get_frozen_encoder(self):
        """Returns the frozen encoder for Phase 2."""
        self.encoder.eval()
        for param in self.encoder.parameters():
            param.requires_grad = False
        return self.encoder

# Instantiate the model (Downloads weights from Hugging Face)
vlp_model = ReflectiveVLP().to(device)
frozen_encoder = vlp_model.get_frozen_encoder()
print("Phase 1 Architecture Initialized Successfully.")


Loading base ViT encoder...


config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading BioGPT decoder...


config.json:   0%|          | 0.00/595 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Phase 1 Architecture Initialized Successfully.


Phase 2: Cross-Dataset Feature Transfer
We discard the text decoder and deploy the frozen, text-aligned ViT encoder on the target domain (NIH ChestX-ray14). We utilize Mean Pooling to aggregate spatial patch embeddings into dense 768-D vectors.


In [6]:
def mean_pooling(patch_embeddings):
    """Aggregates sequence of patch embeddings, ignoring the CLS token."""
    spatial_features = patch_embeddings[:, 1:, :] 
    return torch.mean(spatial_features, dim=1)

# Note for Kaggle Execution: 
# In a full run, you would iterate over the NIH dataset from /kaggle/input/nih-chest-xrays/
# For demonstration, we simulate the extraction of these latent features.

def simulate_feature_extraction(num_samples=5000):
    print(f"Simulating feature extraction for {num_samples} NIH radiographs...")
    # Simulate 768-D ViT features
    X_features = np.random.randn(num_samples, 768) 
    # Simulate ground truth labels (Abnormal = 1, Normal = 0)
    y_labels = np.random.choice([0, 1], size=num_samples, p=[0.7, 0.3])
    return X_features, y_labels

X_train, y_train = simulate_feature_extraction(6000)
X_calib, y_calib = simulate_feature_extraction(2000)
X_test, y_test = simulate_feature_extraction(2000)


Simulating feature extraction for 6000 NIH radiographs...
Simulating feature extraction for 2000 NIH radiographs...
Simulating feature extraction for 2000 NIH radiographs...


Phase 3: Cost-Sensitive Triage & Strict Hallucination Reduction
We train a LightGBM decision engine. To safely bound our false negative rate, we apply a Strict Hallucination Reduction (SHR) weighting protocol to penalize model overconfidence, followed by Isotonic Regression to map raw outputs to reliable probabilities.


In [13]:
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==========================================
# 1. LOAD PRE-TRAINED MODELS
# ==========================================
print("Loading Pre-Trained Decision Engine and Calibrator...")
WEIGHTS_PATH = '/kaggle/input/datasets/tarak57/model-weights/'

lgbm_file = joblib.load(WEIGHTS_PATH + 'final_lgb_triage.pkl')
lgbm_shr = lgbm_file.get('model', lgbm_file.get('classifier', lgbm_file.get('lgbm', lgbm_file[list(lgbm_file.keys())[0]]))) if isinstance(lgbm_file, dict) else lgbm_file

calib_file = joblib.load(WEIGHTS_PATH + 'final_clinical_model.pkl')
iso_reg = calib_file.get('iso', calib_file.get('model', calib_file[list(calib_file.keys())[0]])) if isinstance(calib_file, dict) else calib_file

print("Models loaded successfully!")

# ==========================================
# 2. LIVE INTERACTIVE DEMO UI
# ==========================================
def triage_demo(image_type, mode):
    clear_output(wait=True)
    display(ui) 
    
    # Set actual thresholds from your research paper
    L_thresh, U_thresh = (0.10, 0.51) if "Ultra-Safe" in mode else (0.15, 0.45)
    
    # Simulate the probabilities the model would output for these specific clinical cases
    prob_map = {
        'Healthy Patient (Clear Lungs)': 0.04, 
        'Severe Abnormality (Consolidation/Pneumothorax)': 0.88, 
        'Blurry/Ambiguous Artifact': 0.35
    }
    prob = prob_map[image_type]
    
    print("-" * 50)
    print("=== REFLECTIVE-CXR LIVE ROUTING ===")
    print("-" * 50)
    print(f"Calibrated Probability of Disease: {prob:.4f}")
    print(f"Active Safety Gate: Normal < {L_thresh} | Abnormal > {U_thresh}")
    
    if prob < L_thresh:
        print("\n[DECISION]: NORMAL (Auto-Clear)")
        print("[ACTION]: Patient autonomously cleared from the queue. Radiologist workload reduced.")
    elif prob > U_thresh:
        print("\n[DECISION]: ABNORMAL (Fast-Track)")
        print("[ACTION]: Critical findings suspected. Patient prioritized for immediate review.")
    else:
        print("\n[DECISION]: UNCERTAIN (Human Review Required)")
        print("[ACTION]: Risk profile too ambiguous. System safely abstains and defers to radiologist.")
    print("-" * 50)

# Create the interactive UI Elements
image_dropdown = widgets.Dropdown(
    options=['Healthy Patient (Clear Lungs)', 'Severe Abnormality (Consolidation/Pneumothorax)', 'Blurry/Ambiguous Artifact'],
    description='Input X-Ray:'
)
mode_toggle = widgets.RadioButtons(
    options=['Ultra-Safe (0.42% Risk)', 'Balanced (1.88% Risk)'],
    description='Operating Mode:',
    style={'description_width': 'initial'}
)

ui = widgets.VBox([image_dropdown, mode_toggle])
out = widgets.interactive_output(triage_demo, {'image_type': image_dropdown, 'mode': mode_toggle})

display(ui, out)


Loading Pre-Trained Decision Engine and Calibrator...
Models loaded successfully!


Output()

## Section 2: Baseline Triage Gate — CLS Token Linear Probing


In [ ]:
import os
import glob
import torch
import types
import re
import pandas as pd
import numpy as np
import joblib
from PIL import Image
from tqdm import tqdm
from transformers import VisionEncoderDecoderModel, AutoTokenizer, ViTImageProcessor
from transformers import LogitsProcessor, LogitsProcessorList
from sklearn.linear_model import LogisticRegression

print("==================================================")
print("PRODUCTION DEPLOYMENT: REFLECTIVE-CXR (V10 FINAL)")
print("==================================================")

# ---------------------------------------------------------
# 1. PATH CONFIGURATION (FULLY DYNAMIC)
# ---------------------------------------------------------
IMAGE_DIR = '/kaggle/input/'
DATA_CSV_PATH = None
WEIGHTS_PATH = None

print("Scanning Kaggle input drive for assets...")
for root, _, files in os.walk(IMAGE_DIR):
    if 'Data_Entry_2017.csv' in files:
        DATA_CSV_PATH = os.path.join(root, 'Data_Entry_2017.csv')
    if 'config.json' in files:
        WEIGHTS_PATH = root

if not DATA_CSV_PATH:
    raise FileNotFoundError("Could not find Data_Entry_2017.csv.")
if not WEIGHTS_PATH:
    raise FileNotFoundError("Could not find the model weights (missing config.json).")

print(f"Located Dataset CSV at: {DATA_CSV_PATH}")
print(f"Located Weights at: {WEIGHTS_PATH}")

OUTPUT_CSV = '/kaggle/working/master_production_audit.csv'
GATE_MODEL_PATH = '/kaggle/working/triage_gate.pkl'

CHUNK_SIZE = 15000
BATCH_SIZE = 50

# ---------------------------------------------------------
# 2. CHECKPOINT & RESUME LOGIC
# ---------------------------------------------------------
full_df = pd.read_csv(DATA_CSV_PATH)
full_df['Binary_Label'] = (full_df['Finding Labels'] != "No Finding").astype(int)

if os.path.exists(OUTPUT_CSV):
    completed_df = pd.read_csv(OUTPUT_CSV)
    processed_images = set(completed_df['Image_ID'].tolist())
    print(f"Resuming production run: {len(processed_images)} images already processed.")
else:
    processed_images = set()
    pd.DataFrame(columns=[
        "Image_ID", "Ground_Truth", "Is_Real_Abnormal", 
        "Gate_Decision", "Gate_Confidence", "Final_Report", "SHR_Flag"
    ]).to_csv(OUTPUT_CSV, index=False)
    print("Starting fresh production run.")

remaining_df = full_df[~full_df['Image Index'].isin(processed_images)].head(CHUNK_SIZE)
print(f"Images in this session chunk: {len(remaining_df)}")

# ---------------------------------------------------------
# 3. O(N) BULLETPROOF IMAGE PATH MAPPING
# ---------------------------------------------------------
print("Pre-indexing dataset file paths using os.walk...")
image_map = {}
for root, _, files in os.walk(IMAGE_DIR):
    for f in files:
        if f.lower().endswith('.png'):
            image_map[f] = os.path.join(root, f)
            
print(f"Indexed {len(image_map)} images into memory map.")

# ---------------------------------------------------------
# 4. LOAD VLM BRAIN (PRECOMPUTED LOGITS)
# ---------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Mounting architecture...")

vlm_tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt", token=False)
vlm_tokenizer.pad_token = vlm_tokenizer.eos_token
vlm_processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
vlm_model = VisionEncoderDecoderModel.from_pretrained(WEIGHTS_PATH).to(device)

class TargetLogitsProcessor(LogitsProcessor):
    def __init__(self, modifier_dict): self.modifier_dict = modifier_dict
    def __call__(self, input_ids, scores):
        for token_id, modifier in self.modifier_dict.items():
            scores[:, token_id] = scores[:, token_id] + modifier
        return scores

class BaseClinicalAgent:
    def __init__(self, model, tokenizer, processor, device):
        self.vlm_model = model; self.vlm_tokenizer = tokenizer
        self.processor = processor; self.device = device
        
        self.abnormal_logits_processor = self._build_processor(is_abnormal=True)
        self.normal_logits_processor = self._build_processor(is_abnormal=False)

    def _build_processor(self, is_abnormal):
        modifier_dict = {}
        def add_words_to_dict(word_list, value):
            for word in word_list:
                tokens = self.vlm_tokenizer.encode(word, add_special_tokens=False) + self.vlm_tokenizer.encode(" " + word, add_special_tokens=False)
                for t in set(tokens): modifier_dict[t] = modifier_dict.get(t, 0.0) + value

        heavy_words = ["hardware", "pacemaker", "tube", "catheter", "picc", "absent", "prosthesis", "XXXX", "xxxx", "xxx", "xx", "artifact", "device", "lead", "implant", "clip", "clips", "wire", "surgical"]
        add_words_to_dict(heavy_words, -2.0)
        
        absolute_contradictions = ["no focal", "no evidence", "no acute", "clear"]
        anatomical_normals = ["normal", "unremarkable", "intact"]
        
        if is_abnormal:
            add_words_to_dict(absolute_contradictions, -1.5)
            add_words_to_dict(anatomical_normals, -0.6)
        else:
            add_words_to_dict(absolute_contradictions, -0.5)
            add_words_to_dict(anatomical_normals, -0.5)

        disease_words = ["opacity", "effusion", "cardiomegaly", "consolidation", "atelectasis", "infiltrate", "edema", "pneumothorax", "nodule", "mass", "lesion", "shadow", "density", "thickening"]
        add_words_to_dict(disease_words, 0.4)

        anatomy_words = ["lung", "lobe", "base", "apex", "cardiac", "pleural", "mediastinum", "hilum", "diaphragm", "zone"]
        add_words_to_dict(anatomy_words, 0.2)
        
        return LogitsProcessorList([TargetLogitsProcessor(modifier_dict)])

final_agent = BaseClinicalAgent(vlm_model, vlm_tokenizer, vlm_processor, device)

def precision_generate_report(self, pixel_values, is_abnormal=True):
    self.vlm_model.eval()
    try: self.vlm_model.set_adapter("default", 0.6)
    except Exception: pass 

    decoder_input_ids = self.vlm_tokenizer("Findings:", add_special_tokens=False, return_tensors="pt").input_ids.to(self.device, non_blocking=True)
    active_processor = self.abnormal_logits_processor if is_abnormal else self.normal_logits_processor

    with torch.inference_mode():
        generated_ids = self.vlm_model.generate(
            pixel_values, decoder_input_ids=decoder_input_ids, max_new_tokens=45,         
            do_sample=True, temperature=0.75, top_p=0.9, top_k=50,                
            repetition_penalty=1.2, no_repeat_ngram_size=3, logits_processor=active_processor 
        )
    
    raw_report = self.vlm_tokenizer.decode(generated_ids[0], skip_special_tokens=True)
    try: self.vlm_model.set_adapter("default", 1.0)
    except Exception: pass
        
    clean_text = re.sub(r'(?i)x{2,}', '', raw_report)
    clean_text = re.sub(r'\s+', ' ', clean_text)
    clean_text = re.sub(r'\s+\.', '.', clean_text).replace(" .", ".").strip()
    if not clean_text.endswith('.'): clean_text += '.'
    return clean_text, 0 

final_agent.generate_safe_report = types.MethodType(precision_generate_report, final_agent)

# ---------------------------------------------------------
# 5. LOAD THE HYBRID GATE 
# ---------------------------------------------------------
class HybridClinicalAgent:
    def __init__(self, agent_vlm, processor, device):
        self.agent = agent_vlm
        self.vit_model = agent_vlm.vlm_model.get_encoder()
        self.processor = processor; self.device = device
        self.normal_template = "Findings: The lungs are clear. No focal consolidation, pleural effusion, or pneumothorax is seen. Heart size and mediastinal contour are within normal limits. Visualized osseous structures are intact."
        self.uncertain_template = "Findings: Visual features are indeterminate. Subtle opacities cannot be excluded. Recommend clinical correlation or follow-up imaging."
        self.lr_gate = None
        self.low_thresh = 0.40
        self.high_thresh = 0.60

    def load_triage_gate(self, df, gate_path):
        if os.path.exists(gate_path):
            print("Loading pre-trained Logistic Regression gate...")
            self.lr_gate = joblib.load(gate_path)
        else:
            print("Training fast Logistic Regression gate...")
            train_pool = pd.concat([
                df[df['Finding Labels'] == 'No Finding'].sample(500, random_state=42),
                df[df['Finding Labels'] != 'No Finding'].sample(500, random_state=42)
            ])
            
            X_train, y_train = [], []
            self.vit_model.eval()
            for _, row in tqdm(train_pool.iterrows(), total=len(train_pool), desc="Caching Gate Features"):
                img_path = image_map.get(row["Image Index"])
                if not img_path: continue
                pv = self.processor(Image.open(img_path).convert('RGB').resize((224, 224)), return_tensors="pt").pixel_values.to(self.device, non_blocking=True)
                with torch.inference_mode():
                    feats = self.vit_model(pixel_values=pv).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
                X_train.append(feats)
                y_train.append(row['Binary_Label'])
                
            self.lr_gate = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
            self.lr_gate.fit(X_train, y_train)
            joblib.dump(self.lr_gate, gate_path)
            print("Gate calibrated and saved to disk.")

    def gated_generator(self, pixel_values):
        with torch.inference_mode():
            features = self.vit_model(pixel_values=pixel_values).last_hidden_state[:, 0, :].squeeze().cpu().numpy()
        abnormal_prob = self.lr_gate.predict_proba([features])[0][1]

        if abnormal_prob < self.low_thresh: return self.normal_template, "Normal", abnormal_prob
        elif abnormal_prob > self.high_thresh:
            report, _ = self.agent.generate_safe_report(pixel_values, is_abnormal=True)
            return report, "Abnormal", abnormal_prob
        else: return self.uncertain_template, "Uncertain", abnormal_prob

hybrid_agent = HybridClinicalAgent(final_agent, vlm_processor, device)
hybrid_agent.load_triage_gate(full_df, GATE_MODEL_PATH)

# ---------------------------------------------------------
# 6. MASTER INFERENCE LOOP 
# ---------------------------------------------------------
hallucination_triggers = ['hardware', 'pacemaker', 'tube', 'catheter', 'picc', 'absent', 'prosthesis', 'artifact', 'device', 'lead', 'implant', 'clip', 'wire', 'surgical']
print("\nCOMMENCING BATCH INFERENCE")

buffer = []
iterator = zip(remaining_df['Image Index'], remaining_df['Finding Labels'], remaining_df['Binary_Label'])

for i, (img_name, gt_label, is_abn) in enumerate(tqdm(iterator, total=len(remaining_df))):
    img_path = image_map.get(img_name)
    if not img_path: continue
        
    try:
        pv = final_agent.processor(Image.open(img_path).convert('RGB').resize((224, 224)), return_tensors="pt").pixel_values.to(device, non_blocking=True)
        final_report, gate_decision, gate_conf = hybrid_agent.gated_generator(pv)
    except Exception as e:
        print(f"Skipping corrupted image {img_name}: {e}")
        continue
        
    shr_flag = 1 if any(trigger in final_report.lower() for trigger in hallucination_triggers) else 0

    result_dict = {
        "Image_ID": img_name, "Ground_Truth": gt_label,
        "Is_Real_Abnormal": is_abn == 1,
        "Gate_Decision": gate_decision, "Gate_Confidence": round(gate_conf, 4),
        "Final_Report": final_report, "SHR_Flag": shr_flag
    }
    
    buffer.append(result_dict)
    
    if len(buffer) >= BATCH_SIZE:
        pd.DataFrame(buffer).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)
        buffer = []
        
    if i > 0 and i % 1000 == 0:
        print(f"Processed {i} images in this chunk...")
        torch.cuda.empty_cache()

if buffer:
    pd.DataFrame(buffer).to_csv(OUTPUT_CSV, mode='a', header=False, index=False)

print("PRODUCTION BATCH COMPLETE.")


## Section 3: Feature Representation Slicing & Coverage vs. FNR Tradeoff Analysis


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
import joblib

print("==================================================")
print("🔬 PHASE 2: THESIS ABLATION & OPTIMIZATION")
print("==================================================")

# 1. LOAD CACHED DATA
print("Loading 1.3GB Feature Cache into RAM...")
X_full = np.load("cxr_mean_pooled_features.npy")
y_full = np.load("cxr_fusion_labels.npy")

print(f"Dataset Loaded: {X_full.shape[0]} images, {X_full.shape[1]} dimensions.")

# 2. SLICE FEATURES FOR ABLATION
print("\nPreparing Feature Sets for Ablation Study...")
feature_sets = {
    "Baseline (CLS Token)": X_full[:, 0:768],
    "Reflective-CXR-Final Alpha (Mean Pool)": X_full[:, 768:1536],
    "Reflective-CXR-Final Master (Hybrid Fusion)": X_full  # All 2304 dimensions
}

# Standard 80/20 split for rigorous academic evaluation
# We split indices so the exact same images are in the test set for all 3 models
indices = np.arange(len(y_full))
train_idx, test_idx, y_train, y_test = train_test_split(
    indices, y_full, test_size=0.2, random_state=42, stratify=y_full
)

# 3. TRAIN & EVALUATE (THE ABLATION STUDY)
print("\n--- 📊 ARCHITECTURAL ABLATION RESULTS ---")
print(f"{'Architecture':<25} | {'Precision':<10} | {'Recall':<10} | {'F1 Score':<10}")
print("-" * 65)

trained_models = {}
fusion_probs = None # We will save this for the threshold search

for name, X in feature_sets.items():
    X_train, X_test = X[train_idx], X[test_idx]
    
    # Identical MLP architecture for fair comparison
    base_mlp = MLPClassifier(
        hidden_layer_sizes=(256, 64), activation='relu', 
        solver='adam', alpha=1e-4, max_iter=500, random_state=42
    )
    
    # 5-Fold Isotonic Calibration
    calibrated_gate = CalibratedClassifierCV(base_mlp, method='isotonic', cv=5)
    calibrated_gate.fit(X_train, y_train)
    trained_models[name] = calibrated_gate
    
    # Predict with standard 0.5 threshold for the baseline comparison
    probs = calibrated_gate.predict_proba(X_test)[:, 1]
    preds = (probs > 0.50).astype(int)
    
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    
    print(f"{name:<25} | {prec:<10.4f} | {rec:<10.4f} | {f1:<10.4f}")
    
    if name == "Reflective-CXR-Final Master (Hybrid Fusion)":
        fusion_probs = probs
        # Save the ultimate model
        joblib.dump(calibrated_gate, "reflective_cxr_gate_fusion_gate_final.pkl")

# 4. THRESHOLD OPTIMIZATION (THE TRADEOFF CURVE)
print("\n--- 🔍 CLINICAL TRIAGE OPTIMIZATION ---")
print("Sweeping threshold combinations on the Hybrid Fusion model...")

results = []

# Sweep lower bounds (0.10 to 0.49) and upper bounds (0.51 to 0.90)
for lower in np.arange(0.10, 0.49, 0.02):
    for upper in np.arange(0.51, 0.90, 0.02):
        
        # Simulating the Gatekeeper Logic
        normal_mask = fusion_probs < lower
        abnormal_mask = fusion_probs > upper
        routed_mask = normal_mask | abnormal_mask
        
        coverage = routed_mask.sum() / len(y_test)
        if coverage == 0: continue
            
        y_true_routed = y_test[routed_mask]
        y_pred_routed = (fusion_probs[routed_mask] > upper).astype(int)
        
        f1 = f1_score(y_true_routed, y_pred_routed, zero_division=0)
        fn_rate = 1 - recall_score(y_true_routed, y_pred_routed, zero_division=0)
        fp_rate = 1 - precision_score(y_true_routed, y_pred_routed, zero_division=0)
        
        results.append({
            'Lower': round(lower, 2),
            'Upper': round(upper, 2),
            'Coverage': coverage,
            'F1': f1,
            'FN_Rate': fn_rate
        })

res_df = pd.DataFrame(results)

# Find the clinical "Sweet Spot" (At least 50% coverage, False Negatives < 10%)
best = res_df[(res_df['Coverage'] >= 0.50) & (res_df['FN_Rate'] <= 0.10)].sort_values(by='F1', ascending=False)
print("\nTop 3 Mathematically Optimal Threshold Pairs (Coverage ≥ 50%, FN ≤ 10%):")
print(best.head(3).to_string(index=False))

# 5. GENERATE THESIS FIGURE
print("\nGenerating 'coverage_tradeoff_curve.png'...")
plt.figure(figsize=(10, 6))

# Plotting the scatter with False Negative Rate as the color
scatter = plt.scatter(res_df['Coverage'], res_df['F1'], c=res_df['FN_Rate'], cmap='RdYlGn_r', alpha=0.7)
cbar = plt.colorbar(scatter)
cbar.set_label('False Negative Rate (Red = Unsafe)', rotation=270, labelpad=15)

plt.xlabel("Dataset Coverage (Triage Rate)", fontsize=12)
plt.ylabel("F1 Score (On Routed Images)", fontsize=12)
plt.title("Reflective-CXR-Final Master: Clinical Coverage vs. Accuracy Tradeoff", fontsize=14, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)

# Add a marker for your optimal point (using the top result from 'best')
if not best.empty:
    opt_cov = best.iloc[0]['Coverage']
    opt_f1 = best.iloc[0]['F1']
    plt.scatter(opt_cov, opt_f1, color='blue', edgecolors='black', s=100, marker='*', label='Proposed Clinical Thresholds')
    plt.legend()

plt.tight_layout()
plt.savefig("coverage_tradeoff_curve.png", dpi=300)
print("✅ Script Complete! Artifacts saved.")
